# plife_matcher — learn `m(t) -> j`, don't predict the response

## Why this file exists
Every model in this project predicted a **response** from a gene's features. All
of them lost to closed-form ridge, and ridge lost to predicting the H1 mean.
Then the oracle was measured, honestly:

```
  random corpus row                                    0.5117
  nearest neighbour on X (co-response)                 0.5136
  ridge                                                0.5417
  our best TRAINED operator                            0.5687
  nearest neighbour on DEPMAP, untrained               0.5998
  DepMap, blend of top-3, untrained                    0.6118
  HONEST ORACLE — best single corpus row               0.9103
```

The oracle is real: the matching row is chosen using a random half of the genes
and scored on the **other** half, which never entered the choice. Naive oracle
0.9638, honest 0.9103, so selection inflation is only +0.0535 — and on synthetic
data where no true match exists the same control collapses to 0.5145.

**So the answer is already in the corpus for essentially every gene, and we score
0.57 because we cannot identify which row.** That is a retrieval problem, and it
has never been trained. Attention came closest and was optimising the wrong
thing: it learned to *reproduce the response through a soft average*, not to
*identify the match*.

## Two measured facts that shape the design

**The match is a SET, not a needle.** The two half-gene oracle choices agree on
the same row only 23.5% of the time — 1,850x chance — yet both score ~0.91. And
blending the top-3 beats the top-1 (0.6118 vs 0.5987). So the target is a
neighbourhood, and the loss should be listwise, not a 7,892-way argmax.

**Channels must be combined by learning, not by hand.** X + DepMap at equal
weight (0.5844) is *worse* than DepMap alone (0.5998). Fixed weights hurt.

## The blind spot, stated honestly
`X` is identically zero for 10,499 of 18,080 genes (58.1%) and for 21.5% of the
held-out set, because it describes a gene only as a RESPONDER. DepMap describes
it as a TARGET and covers 94.9% of those genes — but their essentiality profiles
are 1.5x flatter (std 0.133 vs 0.199). Genes that never respond in K562 are also
genes whose knockout does little across 1,208 cell lines; the two silences are
the same silence. Both channels sit at chance there (0.4993, 0.4871).

> CAUTION: the 0.9516 oracle previously quoted for those genes came from the
> NAIVE oracle. The split-gene control was only ever computed over all genes
> together. Cell 8 re-measures it honestly on that subset. If it collapses, their
> responses are mostly noise and there is nothing there to retrieve.

In [ ]:
import sys, platform, math, time, json, os, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as Fn
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": .25, "axes.spines.top": False, "axes.spines.right": False})
DEV = "cuda"
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU"
print(f"torch {torch.__version__}  {torch.cuda.get_device_name(0)}  "
      f"{torch.cuda.mem_get_info()[0]/1e9:.1f} GB free")

CFG = dict(
    path="/kaggle/input/datasets/ericdu847/plife-data-npz/plife_data.npz", ctx="k562",
    channels=("x", "depmap"),      # ablate by dropping entries
    n_ref=2048, d_model=512, d_emb=256, n_blocks=3, drop=0.1,
    topk_train=8,                  # size of the oracle positive SET
    topk_eval=3,                   # measured best on the untrained retrieval
    temp=0.05,
    steps=3000, batch=128, lr=1e-3, wd=1e-2,
    eval_every=250, chunk=512, val_frac=0.15, seed=0,
)
torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=1))

## 1. Load — responses, co-response features, DepMap

In [ ]:
def resolve_npz(p):
    if os.path.isfile(p): return p
    c = sorted(glob.glob(os.path.join(p, "**", "*.npz"), recursive=True)) if os.path.isdir(p) else []
    c += sorted(glob.glob("/kaggle/input/**/*.npz", recursive=True))
    c = [x for i, x in enumerate(c) if x not in c[:i]]
    nm = [x for x in c if os.path.basename(x) == "plife_data.npz"]
    if not (nm or c): raise FileNotFoundError(f"no .npz under {p!r}")
    return max(nm or c, key=os.path.getsize)

CFG["path"] = resolve_npz(CFG["path"])
z = np.load(CFG["path"], allow_pickle=True)
assert "depmap" in z, ("this bundle has no DepMap. re-run pack_for_kaggle.py with "
                       "CRISPRGeneEffect.csv present, then re-upload.")
genes = [str(g) for g in z["genes"]]; gpos = {g: i for i, g in enumerate(genes)}; G = len(genes)
p_ = [str(x) for x in z[f"{CFG['ctx']}_perts"]]
k_ = np.array([i for i, q in enumerate(p_) if q in gpos])
D = torch.from_numpy(z[f"{CFG['ctx']}_delta"][k_]).to(DEV).float()
tgt = torch.from_numpy(np.array([gpos[p_[i]] for i in k_])).long().to(DEV)
rng = np.random.default_rng(CFG["seed"]); pr = rng.permutation(len(k_))
cut = int((1 - CFG["val_frac"]) * len(k_))
fit = torch.from_numpy(np.sort(pr[:cut])).long().to(DEV)
val = torch.from_numpy(np.sort(pr[cut:])).long().to(DEV)
print(f"{len(k_)} perturbations x {G} genes   fit {len(fit)}  val {len(val)}")

with torch.no_grad():
    Xc = D[fit] - D[fit].mean(0, keepdim=True)
    Xc = Xc / Xc.norm(dim=0, keepdim=True).clamp_min(1e-9)
    REF = torch.topk(D[fit].var(0), CFG["n_ref"]).indices.sort().values
    X = Xc.T @ Xc[:, REF]
    X = X / X.norm(dim=1, keepdim=True).clamp_min(1e-9)
    del Xc
    DM = torch.from_numpy(z["depmap"]).to(DEV).float()
    DM = DM - DM.mean(1, keepdim=True)
    DM = DM / DM.norm(dim=1, keepdim=True).clamp_min(1e-9)
    blind = X.norm(dim=1) < 1e-6

CH = {"x": X, "depmap": DM}
FEAT = torch.cat([CH[c] for c in CFG["channels"]], 1)
print(f"channels {CFG['channels']} -> FEAT {tuple(FEAT.shape)}")
print(f"genes blind to X: {int(blind.sum()):,} ({100*blind.float().mean():.1f}%);  "
      f"of held-out targets: {100*blind[tgt[val]].float().mean():.1f}%")

## 2. Metrics — unchanged, so numbers stay comparable to every prior notebook

In [ ]:
Mv = torch.ones(1, G, device=DEV); Mv[0, tgt[val]] = 0.0
Yv = D[val]
bv = blind[tgt[val]]

def prep(A, m):
    a = A * m; a = a - a.mean(1, keepdim=True)
    return a / a.norm(dim=1, keepdim=True).clamp_min(1e-12)

def _ranks(S): return 1.0 - (S > S.diagonal().unsqueeze(1)).sum(1).float() / max(len(S) - 1, 1)

def rank_deb(P, per_row=False):
    S = prep(P, Mv) @ prep(Yv, Mv).T
    r = _ranks(S - S.mean(0, keepdim=True))
    return r if per_row else r.mean().item()

def cos_raw(P):
    a, b = P * Mv, Yv * Mv
    return ((a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1)).clamp_min(1e-12)).mean().item()

def blend(S, kk):
    """Top-k soft retrieval. Chunked: 1393 x k x 18080 is ~1 GB at k=10."""
    v, i = S.topk(kk, dim=1)
    w = torch.softmax(v / CFG["temp"], 1)
    out = torch.zeros(len(S), G, device=DEV)
    for s0 in range(0, len(S), CFG["chunk"]):
        e = min(s0 + CFG["chunk"], len(S))
        out[s0:e] = (D[fit[i[s0:e]]] * w[s0:e].unsqueeze(2)).sum(1)
    return out

def report(nm, P):
    r = rank_deb(P, per_row=True)
    print(f"  {nm:<40}{r.mean():>9.4f}{r[~bv].mean():>10.4f}{r[bv].mean():>12.4f}{cos_raw(P):>9.4f}")
    return r

## 3. Untrained baselines — the bar the matcher must clear

In [ ]:
print(f"  {'method':<40}{'ALL':>9}{'X-OK':>10}{'X-BLIND':>12}{'cosine':>9}")
with torch.no_grad():
    BASE = {}
    BASE["random corpus row"] = report(
        "random corpus row", D[fit[torch.randint(len(fit), (len(val),), device=DEV)]])
    for nm, F_ in (("X (co-response)", X), ("DepMap (essentiality)", DM)):
        S = F_[tgt[val]] @ F_[tgt[fit]].T
        BASE[f"NN {nm}"] = report(f"NN, {nm}", D[fit[S.argmax(1)]])
    Sdm = DM[tgt[val]] @ DM[tgt[fit]].T
    BASE["DepMap top-3"] = report("DepMap, blend of top-3", blend(Sdm, 3))
BAR = max(v.mean().item() for v in BASE.values())
print(f"\n  BAR (best untrained) {BAR:.4f}   |   trained operator 0.5687   |   oracle 0.9103")

## 4. Oracle targets — free supervision, split-gene selected

For every TRAINING gene the correct match can be computed leave-one-out. The
selection uses a random half of the genes only; using all of them would teach the
matcher to reproduce the noise in each response rather than its biology. That is
the same discipline that makes the 0.9103 trustworthy.

The target is the top-`topk_train` SET, because the two half-gene choices agree
only 23.5% of the time while both score ~0.91 — many rows are good.

In [ ]:
with torch.no_grad():
    gA = torch.zeros(G, dtype=torch.bool, device=DEV)
    gA[torch.randperm(G, device=DEV)[:G // 2]] = True
    mA = torch.ones(1, G, device=DEV); mA[0, tgt[fit]] = 0.0
    mA = mA * gA.float().unsqueeze(0)              # choose on half A only
    Yf = prep(D[fit], mA)
    POS = torch.zeros(len(fit), CFG["topk_train"], dtype=torch.long, device=DEV)
    for s0 in range(0, len(fit), CFG["chunk"]):
        e = min(s0 + CFG["chunk"], len(fit))
        S = Yf[s0:e] @ Yf.T
        S[torch.arange(e - s0, device=DEV), torch.arange(s0, e, device=DEV)] = -9  # leave-one-out
        POS[s0:e] = S.topk(CFG["topk_train"], dim=1).indices
    del Yf
print(f"oracle positive sets: {tuple(POS.shape)}  (indices into fit, chosen on gene-half A)")

# sanity: do these targets actually predict, on the half that never chose them?
with torch.no_grad():
    mB = torch.ones(1, G, device=DEV); mB[0, tgt[fit]] = 0.0
    mB = mB * (~gA).float().unsqueeze(0)
    yb = prep(D[fit], mB)
    c_pos = (yb.unsqueeze(1) * yb[POS]).sum(2).mean().item()
    rnd = torch.randint(len(fit), POS.shape, device=DEV)
    c_rnd = (yb.unsqueeze(1) * yb[rnd]).sum(2).mean().item()
    print(f"  cosine on the UNUSED gene half:  oracle set {c_pos:+.4f}   random set {c_rnd:+.4f}"
          f"   -> {'targets are real' if c_pos > c_rnd + .05 else 'TARGETS ARE NOISE — STOP'}")

## 5. The matcher

One embedding per gene from the concatenated channels; similarity is a dot
product; retrieval is top-k; prediction is a softmax blend of the retrieved
**observed** responses. The network never emits a response — it only decides
what to look at.

In [ ]:
class Block(nn.Module):
    def __init__(s, d, p):
        super().__init__(); s.n = nn.LayerNorm(d)
        s.f = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(p), nn.Linear(2 * d, d))
    def forward(s, x): return x + s.f(s.n(x))

class Matcher(nn.Module):
    def __init__(s, d_in, cfg):
        super().__init__()
        s.inp = nn.Linear(d_in, cfg["d_model"])
        s.trunk = nn.Sequential(*[Block(cfg["d_model"], cfg["drop"]) for _ in range(cfg["n_blocks"])])
        s.out = nn.Sequential(nn.LayerNorm(cfg["d_model"]), nn.Linear(cfg["d_model"], cfg["d_emb"]))
        s.logit_scale = nn.Parameter(torch.tensor(math.log(10.0)))
    def emb(s, F_):
        e = s.out(s.trunk(s.inp(F_)))
        return e / e.norm(dim=1, keepdim=True).clamp_min(1e-9)
    def sim(s, qa, qb):
        return (s.emb(qa) @ s.emb(qb).T) * s.logit_scale.exp()

net = Matcher(FEAT.shape[1], CFG).to(DEV)
print(f"{sum(q.numel() for q in net.parameters() if q.requires_grad):,} trainable parameters")

## 6. Self-test before real data

In [ ]:
def selftest():
    m = Matcher(FEAT.shape[1], CFG).to(DEV)
    f = torch.randn(64, FEAT.shape[1], device=DEV)
    e = m.emb(f); assert e.shape == (64, CFG["d_emb"])
    assert torch.allclose(e.norm(dim=1), torch.ones(64, device=DEV), atol=1e-4), "not unit norm"
    s = m.sim(f, torch.randn(200, FEAT.shape[1], device=DEV)); assert s.shape == (64, 200)
    s.sum().backward()
    assert m.inp.weight.grad.abs().sum() > 0, "no gradient into the encoder"
    print(f"  emb{tuple(e.shape)} unit-norm OK   sim{tuple(s.shape)}   "
          f"|d inp| {m.inp.weight.grad.abs().mean():.2e}")
    del m; torch.cuda.empty_cache()
selftest(); print("self-test passed")

## 7. Train — listwise cross-entropy onto the oracle SET

In [ ]:
opt = torch.optim.AdamW(net.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=CFG["steps"] + 1)

@torch.no_grad()
def gate():
    net.eval()
    S = net.sim(FEAT[tgt[val]], FEAT[tgt[fit]])
    P = blend(S, CFG["topk_eval"])
    Ssh = net.sim(FEAT[tgt[val]][torch.randperm(len(val), device=DEV)], FEAT[tgt[fit]])
    Psh = blend(Ssh, CFG["topk_eval"])
    net.train()
    return P, rank_deb(P, per_row=True), rank_deb(Psh)

hist, best, best_step = [], -9.0, -1
t0 = time.time(); net.train()
for step in range(CFG["steps"] + 1):
    b = torch.randint(len(fit), (CFG["batch"],), device=DEV)
    S = net.sim(FEAT[tgt[fit[b]]], FEAT[tgt[fit]])          # (batch, n_fit)
    S[torch.arange(len(b), device=DEV), b] = -1e4           # never retrieve yourself
    logp = torch.log_softmax(S, 1)
    loss = -logp.gather(1, POS[b]).mean()                   # listwise CE onto the SET
    opt.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
    opt.step(); sched.step()

    if step % CFG["eval_every"] == 0:
        P, r, sh = gate()
        hist.append((step, loss.item(), r.mean().item(), sh, r[bv].mean().item()))
        flag = ""
        if r.mean().item() > best:
            best, best_step = r.mean().item(), step
            torch.save({"net": net.state_dict(), "cfg": CFG, "step": step}, "plife_matcher_best.pt")
            flag = " <- best"
        print(f"step {step:>5}  loss {loss.item():>6.3f}  debiased {r.mean():.4f} "
              f"(bar {BAR:.4f}, oracle 0.9103)  blind {r[bv].mean():.4f}  shuf {sh:.4f}{flag}"
              + ("   ** SHUF HIGH **" if sh > 0.53 else ""))
print(f"\n[{time.time()-t0:.0f}s] best {best:.4f} @ step {best_step}")

## 8. Result, blind-subset honest oracle, and bootstrap

In [ ]:
net.load_state_dict(torch.load("plife_matcher_best.pt")["net"]); net.eval()
print(f"  {'method':<40}{'ALL':>9}{'X-OK':>10}{'X-BLIND':>12}{'cosine':>9}")
for nm, r in BASE.items():
    print(f"  {nm:<40}{r.mean():>9.4f}{r[~bv].mean():>10.4f}{r[bv].mean():>12.4f}")
with torch.no_grad():
    S = net.sim(FEAT[tgt[val]], FEAT[tgt[fit]])
    for kk in (1, 3, 10):
        r_m = report(f"MATCHER, blend of top-{kk}", blend(S, kk))

# the blind-subset oracle, measured honestly for the first time
with torch.no_grad():
    hA = torch.zeros(G, dtype=torch.bool, device=DEV)
    hA[torch.randperm(G, device=DEV)[:G // 2]] = True
    ma = Mv * hA.float().unsqueeze(0); mb = Mv * (~hA).float().unsqueeze(0)
    ya = prep(Yv, ma)
    bi = torch.zeros(len(val), dtype=torch.long, device=DEV)
    bvl = torch.full((len(val),), -9.0, device=DEV)
    for s0 in range(0, len(fit), 1024):
        c = ya @ prep(D[fit[s0:s0 + 1024]], ma).T
        v_, i_ = c.max(1); u = v_ > bvl; bvl[u] = v_[u]; bi[u] = i_[u] + s0
    Sb = prep(D[fit[bi]], mb) @ prep(Yv, mb).T
    r_or = _ranks(Sb - Sb.mean(0, keepdim=True))
print(f"\n  HONEST oracle (chosen on gene-half A, scored on B)")
print(f"    all {r_or.mean():.4f}    X-OK {r_or[~bv].mean():.4f}    X-BLIND {r_or[bv].mean():.4f}")
if r_or[bv].mean() < 0.6:
    print("    -> the blind subset has NO retrievable match. their responses are")
    print("       mostly noise; the 0.9516 previously quoted was selection artifact.")

def boot(d, reps=10000):
    d = d.detach().cpu().numpy(); rr = np.random.default_rng(0)
    m = d[rr.integers(0, len(d), (reps, len(d)))].mean(1)
    return d.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)
for nm, ref in (("best untrained baseline", max(BASE.values(), key=lambda v: v.mean())),):
    mu, lo, hi = boot(r_m - ref)
    print(f"\n  matcher vs {nm}: {mu:+.4f} [{lo:+.4f}, {hi:+.4f}]"
          + ("" if lo > 0 else "  <- NOT SIGNIFICANT"))

H = np.array(hist)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(H[:, 0], H[:, 1], color="#444"); ax[0].set(xlabel="step", ylabel="listwise CE",
                                                      title="matching loss")
ax[1].axhspan(.48, .52, color="#bbb", alpha=.35, zorder=0)
ax[1].plot(H[:, 0], H[:, 2], lw=1.6, color="#1b6ca8", label="matcher")
ax[1].plot(H[:, 0], H[:, 4], lw=1.1, color="#7a5195", label="X-blind subset")
ax[1].plot(H[:, 0], H[:, 3], lw=1.1, color="#c1121f", ls="--", label="shuffled query")
ax[1].axhline(BAR, color="#e08b00", ls=":", label=f"untrained bar {BAR:.4f}")
ax[1].axhline(0.9103, color="#2a9d3f", ls=":", label="honest oracle 0.9103")
ax[1].set(xlabel="step", ylabel="debiased rank", title="retrieval quality")
ax[1].legend(fontsize=7); plt.tight_layout(); plt.show()

## 9. Where does the oracle match SIT in each channel's ranking?

The matcher tied untrained cosine (-0.0125 [-0.0306, +0.0061]). Training on
oracle targets bought nothing, which says the ceiling is set by the CHANNELS,
not by the matching function. This quantifies that directly.

For each held-out gene, find the oracle-matching corpus row, then ask what rank
that row occupies when the 7,892 candidates are sorted by each channel's
similarity. Chance is rank ~3,946.

```
  oracle row in the TOP FEW HUNDRED -> the information IS in the channel and the
                                       failure is RE-RANKING. shortlist, then
                                       score properly. a cross-encoder over a few
                                       hundred candidates is the standard fix and
                                       was never tried -- ours was a bi-encoder,
                                       one dot product, which cannot model
                                       query-candidate interaction at all.
  oracle row near CHANCE            -> the channels do not know which row matches.
                                       no head on these inputs can recover it and
                                       the only move is a NEW observable.
```
This is the difference between "build a better matcher" and "go find different
data", and it costs one matmul.

In [ ]:
with torch.no_grad():
    hA = torch.zeros(G, dtype=torch.bool, device=DEV)
    hA[torch.randperm(G, device=DEV)[:G // 2]] = True
    ma = Mv * hA.float().unsqueeze(0)
    ya = prep(Yv, ma)
    bi = torch.zeros(len(val), dtype=torch.long, device=DEV)
    bvl = torch.full((len(val),), -9.0, device=DEV)
    for s0 in range(0, len(fit), 1024):
        c = ya @ prep(D[fit[s0:s0 + 1024]], ma).T
        v_, i_ = c.max(1)
        u = v_ > bvl
        bvl[u] = v_[u]
        bi[u] = i_[u] + s0

    rows = []
    S_m = net.sim(FEAT[tgt[val]], FEAT[tgt[fit]])
    for nm, S in (("X (co-response)", X[tgt[val]] @ X[tgt[fit]].T),
                  ("DepMap (essentiality)", DM[tgt[val]] @ DM[tgt[fit]].T),
                  ("trained matcher", S_m)):
        rk = (S > S.gather(1, bi.unsqueeze(1))).sum(1).float()
        rows.append((nm, rk))

n = len(fit)
print(f"  candidates: {n:,}   chance rank {n//2:,}")
print()
print(f"  {'channel':<26}{'median':>9}{'top-10':>9}{'top-100':>9}{'top-1000':>10}")
for nm, rk in rows:
    print(f"  {nm:<26}{rk.median():>9.0f}{100*(rk<10).float().mean():>8.1f}%"
          f"{100*(rk<100).float().mean():>8.1f}%{100*(rk<1000).float().mean():>9.1f}%")
rnd = torch.randint(n, (len(val),), device=DEV).float()
print(f"  {'chance':<26}{rnd.median():>9.0f}{100*(rnd<10).float().mean():>8.1f}%"
      f"{100*(rnd<100).float().mean():>8.1f}%{100*(rnd<1000).float().mean():>9.1f}%")

best_rk = min(rows, key=lambda r: r[1].median())
med = best_rk[1].median().item()
print()
if med < n * 0.05:
    print(f"  => the oracle row is usually in the top {100*med/n:.1f}% ({best_rk[0]}).")
    print("     THE INFORMATION IS IN THE CHANNEL. This is a RE-RANKING problem:")
    print("     shortlist a few hundred candidates, then score them with a")
    print("     cross-encoder. Never tried -- ours was a bi-encoder.")
else:
    print(f"  => the oracle row sits at median rank {med:.0f} of {n:,} "
          f"({100*med/n:.0f}% down the list).")
    print("     THE CHANNELS DO NOT KNOW WHICH ROW MATCHES. No head on these")
    print("     inputs can recover it; the only move is a new observable.")

fig, ax = plt.subplots(figsize=(5.4, 3.4))
for nm, rk in rows:
    v = torch.sort(rk).values.cpu().numpy()
    ax.plot(v, np.linspace(0, 1, len(v)), label=nm)
ax.plot(np.sort(rnd.cpu().numpy()), np.linspace(0, 1, len(val)), "k--", lw=.8, label="chance")
ax.set(xscale="log", xlabel="rank of the ORACLE row in this channel's ordering",
       ylabel="fraction of held-out genes", title="is the answer findable from features?")
ax.legend(fontsize=7.5)
plt.tight_layout()
plt.show()

## 10. Confidence-gated retrieval — exploit the bimodality instead of averaging it

Cell 9's verdict logic was wrong: it thresholded on the MEDIAN, and retrieval
only ever reads the top of the list. The actual shape:

```
  channel                 median   top-10  top-100  top-1000
  X (co-response)           2024    22.0%    23.5%     37.8%
  DepMap                    3016     7.0%    11.2%     24.7%
  trained matcher           1558     5.7%    16.3%     40.6%
  chance                    4025     0.3%     1.0%     11.6%
```

**X puts the oracle row in the top 10 for 22.0% of genes — 73x chance.** And
top-10 22.0% vs top-100 23.5% means almost nothing lands in between: either X
finds it essentially first, or not at all. That is a near-duplicate existing in
the corpus, or not existing.

**The trained matcher destroyed exactly that**, cutting top-10 from 22.0% to
5.7% while improving the median. It optimised the bulk of the ranking, which
retrieval never reads.

So do not average over the two regimes — DETECT which one you are in. If X's
top-1 similarity itself predicts whether its match is real, retrieval can be
gated: take X's match when confident, fall back otherwise. No training.

**The gate threshold is chosen on the FIT set**, never on val. Choosing it on
val is the selection inflation this project has already paid for twice.

In [ ]:
with torch.no_grad():
    # --- confidence signal and per-row scores for each strategy ----------------
    Sx_v = X[tgt[val]] @ X[tgt[fit]].T
    Sd_v = DM[tgt[val]] @ DM[tgt[fit]].T
    conf_v = Sx_v.max(1).values                       # how sure is X?

    r_x = rank_deb(blend(Sx_v, 1), per_row=True)
    r_d = rank_deb(blend(Sd_v, 3), per_row=True)

    # --- choose the threshold on FIT, by leave-one-out retrieval among fit -----
    Sx_f = X[tgt[fit]] @ X[tgt[fit]].T
    Sd_f = DM[tgt[fit]] @ DM[tgt[fit]].T
    eye = torch.arange(len(fit), device=DEV)
    Sx_f[eye, eye] = -9
    Sd_f[eye, eye] = -9
    conf_f = Sx_f.max(1).values
    Mf = torch.ones(1, G, device=DEV)
    Mf[0, tgt[fit]] = 0.0
    yf = prep(D[fit], Mf)

    def rank_fit(S, kk):
        v, i = S.topk(kk, dim=1)
        w = torch.softmax(v / CFG["temp"], 1)
        out = torch.zeros(len(fit), G, device=DEV)
        for s0 in range(0, len(fit), CFG["chunk"]):
            e = min(s0 + CFG["chunk"], len(fit))
            out[s0:e] = (D[fit[i[s0:e]]] * w[s0:e].unsqueeze(2)).sum(1)
        Sm = prep(out, Mf) @ yf.T
        return _ranks(Sm - Sm.mean(0, keepdim=True))

    rf_x, rf_d = rank_fit(Sx_f, 1), rank_fit(Sd_f, 3)
    cand_thr = torch.quantile(conf_f, torch.linspace(0.5, 0.95, 10).to(DEV))
    best_t, best_s = None, -9
    for t_ in cand_thr:
        use = conf_f >= t_
        s_ = torch.where(use, rf_x, rf_d).mean().item()
        if s_ > best_s:
            best_s, best_t = s_, t_.item()
    print(f"  gate threshold chosen on FIT: X-confidence >= {best_t:.4f}"
          f"   (fit score {best_s:.4f})")
    print(f"  it fires on {100*(conf_v>=best_t).float().mean():.1f}% of held-out genes")

    gated = torch.where(conf_v >= best_t, r_x, r_d)

print(f"\n  {'strategy':<40}{'debiased':>10}")
print(f"  {'X top-1 everywhere':<40}{r_x.mean():>10.4f}")
print(f"  {'DepMap top-3 everywhere':<40}{r_d.mean():>10.4f}")
print(f"  {'GATED: X when confident, else DepMap':<40}{gated.mean():>10.4f}")
print(f"  {'trained matcher':<40}{0.6008:>10.4f}")
print(f"  {'honest oracle':<40}{0.9055:>10.4f}")

def boot(d, reps=10000):
    d = d.detach().cpu().numpy()
    rr = np.random.default_rng(0)
    m = d[rr.integers(0, len(d), (reps, len(d)))].mean(1)
    return d.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)
mu, lo, hi = boot(gated - r_d)
tag = "" if lo > 0 else ("  <- NOT SIGNIFICANT" if hi > 0 else "  <- WORSE")
print(f"\n  gated vs DepMap-everywhere: {mu:+.4f} [{lo:+.4f}, {hi:+.4f}]{tag}")

# --- does confidence actually predict correctness? -------------------------
q = torch.quantile(conf_v, torch.linspace(0, 1, 6).to(DEV))
print(f"\n  {'X-confidence bin':<26}{'n':>6}{'X top-1':>10}{'DepMap':>9}{'better':>9}")
rowsC = []
for b in range(5):
    m_ = (conf_v >= q[b]) & ((conf_v <= q[b + 1]) if b == 4 else (conf_v < q[b + 1]))
    if m_.sum() < 5:
        continue
    a_, c_ = r_x[m_].mean().item(), r_d[m_].mean().item()
    rowsC.append((float((q[b] + q[b + 1]) / 2), a_, c_))
    print(f"  {float(q[b]):.3f}-{float(q[b+1]):.3f}{'':<12}{int(m_.sum()):>6}"
          f"{a_:>10.4f}{c_:>9.4f}{'  X' if a_ > c_ else '  DepMap':>9}")

C = np.array(rowsC)
fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.plot(C[:, 0], C[:, 1], "o-", color="#1b6ca8", label="X top-1")
ax.plot(C[:, 0], C[:, 2], "s-", color="#e08b00", label="DepMap top-3")
ax.axvline(best_t, color="#c1121f", ls="--", lw=1, label=f"gate {best_t:.3f} (from fit)")
ax.axhline(.5, color="#888", lw=.8)
ax.set(xlabel="X top-1 similarity (confidence)", ylabel="debiased rank",
       title="does X know when it knows?")
ax.legend(fontsize=7.5)
plt.tight_layout()
plt.show()